# 09 Multi-Accident Dataset Audit

This notebook audits every non-Normal fixed-power accident trajectory in `Operation_csv_data`. Each CSV is one complete `sample_id`; time rows are never randomly split. It does not train the final classifier.

The strict process baseline reuses the README-backed `results/feature_groups.csv` policy from 03 and keeps only its 38 `candidate` variables. It also records event parsing, early-window coverage, category-specific leakage flags, t=0 initial-condition differences, and a trajectory-grouped split plan.

In [1]:
from pathlib import Path
import sys

project_candidates = [Path.cwd(), Path.cwd().parent, Path(r'C:/Users/18205/NPP-Guard')]
PROJECT_ROOT = next(path for path in project_candidates if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from multi_accident import run_audit

In [2]:
result = run_audit(PROJECT_ROOT)
summary = result['summary']
print('Project root:', PROJECT_ROOT)
print('Accident classes:', summary['accident_class_count'])
print('Trajectories:', summary['trajectory_count'])
print('Class-size range:', summary['class_trajectory_count_min'], 'to', summary['class_trajectory_count_max'])
print('Strict process features:', summary['strict_process_feature_count'])
print('Event parsing:', summary['event_parse_success'])
print('Window coverage:', summary['available_window_coverage'])
result['class_summary'][['accident_class', 'trajectory_count', 'available_30s', 'available_60s', 'available_120s', 'first_version_eligible']]

Project root: C:\Users\18205\NPP-Guard
Accident classes: 17
Trajectories: 1216
Class-size range: 1 to 110
Strict process features: 38
Event parsing: {'reports': 1216, 'injection': 1216, 'scram': 662, 'first_protection': 662, 'trajectory_total': 1216}
Window coverage: {'30': {'available_trajectories': 1216, 'coverage_rate': 1.0, 'classes_with_full_coverage': ['ATWS', 'FLB', 'LACP', 'LLB', 'LOCA', 'LOCAC', 'LOF', 'LR', 'MD', 'RI', 'RW', 'SGATR', 'SGBTR', 'SLBIC', 'SLBOC', 'SP', 'TT']}, '60': {'available_trajectories': 1216, 'coverage_rate': 1.0, 'classes_with_full_coverage': ['ATWS', 'FLB', 'LACP', 'LLB', 'LOCA', 'LOCAC', 'LOF', 'LR', 'MD', 'RI', 'RW', 'SGATR', 'SGBTR', 'SLBIC', 'SLBOC', 'SP', 'TT']}, '120': {'available_trajectories': 1198, 'coverage_rate': 0.9851973684210527, 'classes_with_full_coverage': ['ATWS', 'FLB', 'LACP', 'LLB', 'LOCA', 'LOCAC', 'LOF', 'LR', 'MD', 'RW', 'SGATR', 'SGBTR', 'SLBIC', 'SLBOC', 'SP', 'TT']}}


,accident_class,trajectory_count,available_30s,available_60s,available_120s,first_version_eligible
0,ATWS,1,1,1,1,False
1,FLB,100,100,100,100,True
2,LACP,1,1,1,1,False
3,LLB,101,101,101,101,True
4,LOCA,100,100,100,100,True
5,LOCAC,100,100,100,100,True
6,LOF,1,1,1,1,False
7,LR,99,99,99,99,True
8,MD,100,100,100,100,True
9,RI,100,100,100,82,True


In [3]:
print('Potential leakage flags:')
display(result['leakage_flags'].head(30))
print('Initial-condition flags:', summary['initial_condition_audit']['flagged_features'])
display(result['initial_conditions'].query("feature in ['P', 'PWR', 'TAVG', 'LVPZ', 'TSAT', 'VOL']"))

Potential leakage flags:


,feature,readme_definition,group,ml_policy_from_03,accident_class,class_trajectory_count,other_trajectory_count,class_presence_rate,other_presence_rate,class_state_change_rate,other_state_change_rate,class_low_cardinality_rate,other_low_cardinality_rate,risk_type,trigger,confidence_note,potential_label_leakage,auto_removed,recommendation
0,LVCR,Level Core water (M),observable_process,candidate,LOCA,100,1116,1.00,1.000000,0.91,0.080645,0.09,0.919355,potential_label_leakage,class_specific_state_change,review_candidate,True,False,Review before training; audit does not delete ...
1,LVCR,Level Core water (M),observable_process,candidate,LOCAC,100,1116,1.00,1.000000,0.90,0.081541,0.10,0.918459,potential_label_leakage,class_specific_state_change,review_candidate,True,False,Review before training; audit does not delete ...
2,PSGA,Pressure Steam generator A (bar),observable_process,candidate,SLBIC,101,1115,1.00,1.000000,1.00,1.000000,0.00,0.000000,initial_condition_confounding,class_specific_initial_constant,review_candidate,False,False,Review before training; audit does not delete ...
3,PSGB,Pressure Steam generator B (bar),observable_process,candidate,SLBIC,101,1115,1.00,1.000000,1.00,1.000000,0.00,0.000000,initial_condition_confounding,class_specific_initial_constant,review_candidate,False,False,Review before training; audit does not delete ...
4,QMGA,Power SG A heat removal (MW),observable_process,candidate,SLBIC,101,1115,1.00,1.000000,1.00,1.000000,0.00,0.000000,initial_condition_confounding,class_specific_initial_constant,review_candidate,False,False,Review before training; audit does not delete ...
5,QMGB,Power SG B heat removal (MW),observable_process,candidate,SLBIC,101,1115,1.00,1.000000,1.00,1.000000,0.00,0.000000,initial_condition_confounding,class_specific_initial_constant,review_candidate,False,False,Review before training; audit does not delete ...
6,QMWT,PowerTotal megawatt thermal (MW),observable_process,candidate,SLBIC,101,1115,1.00,1.000000,1.00,1.000000,0.00,0.000000,initial_condition_confounding,class_specific_initial_constant,review_candidate,False,False,Review before training; audit does not delete ...
7,RM4,Rad Monitor Aux Building Air (CPM),radiological_or_dose,exclude,LLB,101,1115,1.00,1.000000,1.00,0.000000,0.00,1.000000,potential_label_leakage,class_specific_state_change,review_candidate,True,False,Review before training; audit does not delete ...
8,SGLK,Mass Total Leakage out of SGs (kg),radiological_or_dose,exclude,LACP,1,1215,1.00,0.030453,1.00,0.030453,0.00,0.974486,potential_label_leakage,class_specific_nonzero;class_specific_state_ch...,low_sample_class,True,False,Review before training; audit does not delete ...
9,SGLK,Mass Total Leakage out of SGs (kg),radiological_or_dose,exclude,TT,1,1215,1.00,0.030453,1.00,0.030453,0.00,0.974486,potential_label_leakage,class_specific_nonzero;class_specific_state_ch...,low_sample_class,True,False,Review before training; audit does not delete ...


Initial-condition flags: ['PSGA', 'PSGB', 'QMGA', 'QMGB', 'QMWT', 'TAVG', 'TCA', 'TCB', 'THA', 'THB', 'WFWA', 'WFWB', 'WSTA', 'WSTB']


,feature,accident_class,initial_count,initial_mean,initial_std,initial_min,initial_max,max_pairwise_mean_diff,global_class_mean_max,global_initial_std,initial_condition_flag,class_mean_vs_other_max_diff,class_mean_vs_global_class_median_diff,class_initial_condition_flag
51,LVPZ,ATWS,1,62.000000,NaN,62.000000,62.000000,0.0,62.000000,0.0,False,0.0,0.0,False
52,LVPZ,FLB,100,62.000000,0.0,62.000000,62.000000,0.0,62.000000,0.0,False,0.0,0.0,False
53,LVPZ,LACP,1,62.000000,NaN,62.000000,62.000000,0.0,62.000000,0.0,False,0.0,0.0,False
54,LVPZ,LLB,101,62.000000,0.0,62.000000,62.000000,0.0,62.000000,0.0,False,0.0,0.0,False
55,LVPZ,LOCA,100,62.000000,0.0,62.000000,62.000000,0.0,62.000000,0.0,False,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,VOL,SGBTR,110,289.644989,0.0,289.644989,289.644989,0.0,289.644989,0.0,False,0.0,0.0,False
540,VOL,SLBIC,101,289.644989,0.0,289.644989,289.644989,0.0,289.644989,0.0,False,0.0,0.0,False
541,VOL,SLBOC,100,289.644989,0.0,289.644989,289.644989,0.0,289.644989,0.0,False,0.0,0.0,False
542,VOL,SP,1,289.644989,NaN,289.644989,289.644989,0.0,289.644989,0.0,False,0.0,0.0,False


In [4]:
inventory = result['class_inventory']
assert inventory['sample_id'].is_unique
assert len(inventory) == summary['trajectory_count']
assert summary['strict_process_feature_count'] == 38
assert set(inventory['split']) <= {'train', 'validation', 'test'}
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert not (set(inventory.loc[inventory['split'].eq(left), 'sample_id']) & set(inventory.loc[inventory['split'].eq(right), 'sample_id']))
assert all(Path(PROJECT_ROOT / path).exists() for path in result['output_paths'].values())
print('Split policy: complete trajectory/sample_id groups only')
print('First-version classes:', summary['recommendation']['first_version_eligible_classes'])
print('Deferred small classes:', summary['recommendation']['defer_or_expand_classes'])
print('Recommended windows:', summary['recommendation']['recommended_windows'], 'with 120 s exploratory')
print('Metrics:', summary['recommendation']['metrics'])
print('FULL MULTI-ACCIDENT DATASET AUDIT PASSED')

Split policy: complete trajectory/sample_id groups only
First-version classes: ['FLB', 'LLB', 'LOCA', 'LOCAC', 'LR', 'MD', 'RI', 'RW', 'SGATR', 'SGBTR', 'SLBIC', 'SLBOC']
Deferred small classes: ['ATWS', 'LACP', 'LOF', 'SP', 'TT']
Recommended windows: [30, 60] with 120 s exploratory
Metrics: ['Macro-F1', 'Balanced Accuracy', 'per-class Recall', 'confusion matrix']
FULL MULTI-ACCIDENT DATASET AUDIT PASSED
